# Assignment 3
# Andrew Silveira
# 5077086

### Imports

In [ ]:
import numpy as np
# --- PATCH START ---
# This fixes the "AttributeError: module 'numpy' has no attribute 'bool8'"
# by manually re-adding the missing attribute to NumPy.
if not hasattr(np, 'bool8'):
    np.bool8 = np.bool_
# --- PATCH END ---

import gym
import random
from collections import deque
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

from assignment3_utils import process_frame, transform_reward

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


## DQNCNN Class

The `DQNCNN` class implements the convolutional neural network used to approximate
the Q-function for the Pong environment. It follows the standard DeepMind
architecture for Atari games.

### **Architecture**
- **Input:** Stacked grayscale frames (shape: 4 × 84 × 80)
- **Conv Layer 1:** 32 filters, 8×8 kernel, stride 4, ReLU
- **Conv Layer 2:** 64 filters, 4×4 kernel, stride 2, ReLU
- **Conv Layer 3:** 64 filters, 3×3 kernel, stride 1, ReLU
- **Fully Connected:** 512 units, ReLU
- **Output Layer:** Linear layer producing Q-values for each action

### **Purpose**
This network maps a state (stack of frames) to Q-values for all available
actions, enabling the agent to select the action with the highest expected
future reward.


In [ ]:
class DQNCNN(nn.Module):
    def __init__(self, action_size):
        super(DQNCNN, self).__init__()
        
        # Input: (batch, 4, 84, 80)
        
        self.conv = nn.Sequential(
            nn.Conv2d(4, 32, kernel_size=8, stride=4),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1),
            nn.ReLU()
        )
        
        # Compute conv output size
        
        dummy = torch.zeros(1, 4, 84, 80)
        conv_out = self.conv(dummy).view(1, -1).size(1)
        
        self.fc = nn.Sequential(
            nn.Linear(conv_out, 512),
            nn.ReLU(),
            nn.Linear(512, action_size)
        )
    
    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

## ReplayBuffer

The `ReplayBuffer` class stores past transitions so the agent can sample
mini-batches of experience during training. This breaks temporal correlations
and stabilizes learning.

### **Stored Elements**
Each entry contains:
- `state`
- `action`
- `reward`
- `next_state`
- `done` flag

### **Purpose**
- Provides randomized training samples
- Prevents the network from overfitting to recent transitions
- Enables stable gradient updates

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity=50000):
        self.capacity = capacity
        #self.buffer = []
        #self.position = 0
        self.buffer = deque(maxlen=capacity)
    
    def push(self, state, action, reward, next_state, done):     
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            np.array(states),
            np.array(actions),
            np.array(rewards),
            np.array(next_states),
            np.array(dones)
        )

    def __len__(self):
        return len(self.buffer)

## DQNAgent

The `DQNAgent` class encapsulates the DQN, target network, optimizer, and action
selection logic (epsilon-greedy policy).

### **Responsibilities**
- Choose actions using epsilon-greedy exploration
- Store transitions in the replay buffer
- Perform gradient updates on the Q-network
- Periodically update the target network

### **Key Attributes**
- `policy_net`: Main DQN used for action selection
- `target_net`: Stable target network
- `optimizer`: Adam optimizer
- `epsilon`: Exploration rate

In [ ]:
class DQNAgent:
    def __init__(self, action_size, batch_size=8, gamma=0.95, lr=1e-4):
        self.action_size = action_size
        self.batch_size = batch_size
        self.gamma = gamma

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.policy_net = DQNCNN(action_size).to(self.device)
        self.target_net = DQNCNN(action_size).to(self.device)
        self.target_net.load_state_dict(self.policy_net.state_dict())

        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=lr)
        self.memory = ReplayBuffer()

        # Exploration
        self.epsilon = 1.0
        self.epsilon_min = 0.05
        self.epsilon_decay = 0.995

    def act(self, state):
        if random.random() < self.epsilon:
            return random.randrange(self.action_size)

        state = torch.FloatTensor(state).unsqueeze(0).to(self.device)
        q_values = self.policy_net(state)
        return torch.argmax(q_values).item()

    def train_step(self):
        if len(self.memory) < self.batch_size:
            return

        states, actions, rewards, next_states, dones = self.memory.sample(self.batch_size)

        states = torch.FloatTensor(states).to(self.device)
        next_states = torch.FloatTensor(next_states).to(self.device)
        actions = torch.LongTensor(actions).to(self.device)
        rewards = torch.FloatTensor(rewards).to(self.device)
        dones = torch.FloatTensor(dones).to(self.device)

        q_values = self.policy_net(states)
        next_q_values = self.target_net(next_states)

        q_target = rewards + self.gamma * torch.max(next_q_values, dim=1)[0] * (1 - dones)

        q_pred = q_values.gather(1, actions.unsqueeze(1)).squeeze()

        loss = nn.MSELoss()(q_pred, q_target.detach())

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        # Decay epsilon
        self.epsilon = max(self.epsilon * self.epsilon_decay, self.epsilon_min)

## PongTrainer

The `PongTrainer` class manages the full training loop for the Pong environment.
It handles environment resets, episode iteration, reward tracking, and calling
the agent’s optimization steps.

### **Responsibilities**
- Run episodes of Pong
- Collect rewards and training metrics
- Interact with the agent to select actions and update networks
- Return score histories for plotting and analysis

In [ ]:
class PongTrainer:
    def stack_frames(self, stacked_frames, new_frame, is_new_episode):
        if is_new_episode:
            stacked_frames = deque([new_frame] * 4, maxlen=4)
        else:
            stacked_frames.append(new_frame)

        return np.stack(stacked_frames, axis=0), stacked_frames

    def train_pong(self, batch_size=8, target_update_rate=10, num_episodes=500, render=False):
        #env = gym.make("PongDeterministic-v4")
        env = gym.make("PongDeterministic-v4", render_mode="human" if render else None)
        action_size = env.action_space.n

        agent = DQNAgent(action_size, batch_size=batch_size)

        scores = []
        last5_rewards = []

        for episode in range(num_episodes):
            obs, info = env.reset()
            frame = process_frame(obs, (84, 80))
            stacked_frames = None
            state, stacked_frames = self.stack_frames(stacked_frames, frame, True)

            done = False
            total_reward = 0

            while not done:
                action = agent.act(state)
                next_obs, reward, terminated, truncated, info = env.step(action)
                done = terminated or truncated

                reward = transform_reward(reward)
                next_frame = process_frame(next_obs, (84, 80))
                next_state, stacked_frames = self.stack_frames(stacked_frames, next_frame, False)

                agent.memory.push(state, action, reward, next_state, done)
                agent.train_step()

                state = next_state
                total_reward += reward

            scores.append(total_reward)
            last5_rewards.append(np.mean(scores[-5:]))

            print(f"Episode {episode} | Score: {total_reward} | Avg(5): {last5_rewards[-1]:.2f}", flush=True)

            # Update target network every N episodes
            if episode % target_update_rate == 0:
                agent.target_net.load_state_dict(agent.policy_net.state_dict())

        env.close()
        return scores, last5_rewards

## Training Plots

 - Due to computational constraints and thermal limitations on the hardware used (a consumer laptop without dedicated cooling), I trained the agent for 20 episodes per configuration. While this is not sufficient for full convergence on Pong, it is enough to observe early-stage learning behavior and compare the relative effects of different mini-batch sizes and target network update rates.

In [ ]:
trainer = PongTrainer()

# Experiment 1: Batch size
scores_b8, avg5_b8 = trainer.train_pong(batch_size=8, target_update_rate=10, num_episodes=20)
scores_b16, avg5_b16 = trainer.train_pong(batch_size=16, target_update_rate=10, num_episodes=20)

plt.figure(figsize=(10,5))
plt.plot(scores_b8, label="Batch size 8")
plt.plot(scores_b16, label="Batch size 16")
plt.xlabel("Episode")
plt.ylabel("Score")
plt.title("Effect of Mini-Batch Size on DQN Performance")
plt.legend()
plt.show()
    
# Experiment 2: Target update rate
scores_u3, avg5_u3 = trainer.train_pong(batch_size=8, target_update_rate=3, num_episodes=20)
scores_u10, avg5_u10 = trainer.train_pong(batch_size=8, target_update_rate=10, num_episodes=20)

plt.figure(figsize=(10,5))
plt.plot(scores_u3, label="Update rate = 3")
plt.plot(scores_u10, label="Update rate = 10")
plt.xlabel("Episode")
plt.ylabel("Score")
plt.title("Effect of Target Network Update Rate on DQN Performance")
plt.legend()
plt.show()

Episode 0 | Score: -21.0 | Avg(5): -21.00
Episode 1 | Score: -21.0 | Avg(5): -21.00
Episode 2 | Score: -21.0 | Avg(5): -21.00
Episode 3 | Score: -21.0 | Avg(5): -21.00
Episode 4 | Score: -21.0 | Avg(5): -21.00
Episode 5 | Score: -21.0 | Avg(5): -21.00
Episode 6 | Score: -20.0 | Avg(5): -20.80
Episode 7 | Score: -21.0 | Avg(5): -20.80
Episode 8 | Score: -20.0 | Avg(5): -20.60
Episode 9 | Score: -19.0 | Avg(5): -20.20
Episode 10 | Score: -21.0 | Avg(5): -20.20
Episode 11 | Score: -20.0 | Avg(5): -20.20
Episode 12 | Score: -21.0 | Avg(5): -20.20
Episode 13 | Score: -21.0 | Avg(5): -20.40
Episode 14 | Score: -21.0 | Avg(5): -20.80
Episode 15 | Score: -21.0 | Avg(5): -20.80
Episode 16 | Score: -20.0 | Avg(5): -20.80
Episode 17 | Score: -21.0 | Avg(5): -20.80
Episode 18 | Score: -19.0 | Avg(5): -20.40
Episode 19 | Score: -21.0 | Avg(5): -20.40
Episode 0 | Score: -20.0 | Avg(5): -20.00
Episode 1 | Score: -21.0 | Avg(5): -20.50
Episode 2 | Score: -21.0 | Avg(5): -20.67
Episode 3 | Score: -21.0

## Pong OpenAI Gym

In [ ]:
scores, avg5 = trainer.train_pong(
    batch_size=8,
    target_update_rate=10,
    num_episodes=15,
    render=True
)